# autoresearch-xla: Session 2 (`exp/xla-vendetta`) Kaggle TPU `v5e-8` — 1B-Safe Stable + Full ICI/Muon Optimization

Upgrades `1b-ve` (`1.456B` total / `1.120B` active, Split-Muon ON, MTP OFF) with **real-silicon-verified `exp/xla-vendetta` optimizations** AND **1B-safe stable hyperparameters** (`matrix_lr=0.015, embedding_lr=0.25, unembedding_lr=0.003, scalar_lr=0.1, weight_decay=0.1, warmup_steps=1000` — fixing Session 1's `0.04` LR oscillation):

1. **Cell 1 — Environment (`exp/xla-vendetta`) & Pallas SplashAttention Unlock**: Clones `exp/xla-vendetta` to `/tmp/autoresearch-xla`, unlocks Pallas SplashAttention, and enables all 10 verified MaxText FSDP `LIBTPU_INIT_ARGS`.
2. **Cell 2 — Smart Private HF Hub Checkpoint Health Check**: Connects to `<your-hf-username>/autoresearch-1b-ve`, inspects Session 1's saved `latest/metrics.jsonl` (`val_bpb` + `gnorm`), and **auto-resumes ONLY if the saved checkpoint is healthy** (`val_bpb <= best + 0.015` and `gnorm < 0.8`), otherwise automatically starts clean from step 0 (`--no-resume`) so degraded weights never pollute the 20B run!
3. **Cell 3 — Stage 1 Zero-Waste Preflight Gate (~2 min)**: Benchmarks `vendetta_b4_sac` (`b=4, save_all_compute, splash` — U-only Layer-Sharded Split-Muon, Contracting-Axis FSDP, Replicated Small Keys, Hidden-Sharded Embeds, Aligned Canon-4, S-layer Splash `blk=512`, virtual fused `w_qkv_ag`, zero-`s32`-leak remat + output-reusing QK-norm/sigmoid VJPs). Short-circuits immediately if finite-loss, `< 15,200 MB` HBM, and `>= 85,000 tok/s` pass; auto-falls back to `save_attn` if HBM is tight.
4. **Cell 4 — Stage 2 Immediate Auto-Promote + Real-Silicon XProf Capture**: Trains 8.2h with 1B-safe LRs, capturing a 2-step real-silicon XProf trace (`/kaggle/working/xprof_session2`) and syncing atomic HF Hub checkpoints every 500 steps.
5. **Cell 5 — Post-Run Summary & Curves**: Saves `training_curve.png` and `session_summary.json`.


In [ ]:
import ast
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/OttoApocalypse69/autoresearch-xla.git"
GIT_BRANCH = "exp/xla-vendetta"
REPO_DIR = Path("/tmp/autoresearch-xla")

def run(cmd, cwd=None, env=None, check=True):
    print(f">>> {' '.join(map(str, cmd))}", flush=True)
    return subprocess.run(list(map(str, cmd)), cwd=cwd, env=env, check=check)

# 1. Clone exp/xla-vendetta into /tmp so /kaggle/working stays 100% free for 6.8 GB checkpoints
if (REPO_DIR / "scale" / "pretrain.py").exists():
    run(["git", "-C", str(REPO_DIR), "fetch", "--depth=1", "origin", GIT_BRANCH])
    run(["git", "-C", str(REPO_DIR), "checkout", "-B", GIT_BRANCH, "FETCH_HEAD"])
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run(["git", "clone", "--depth=1", "--branch", GIT_BRANCH, REPO_URL, str(REPO_DIR)])

commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"Repo ready at {REPO_DIR} (branch {GIT_BRANCH}, commit {commit})", flush=True)

# 2. Install lightweight non-JAX deps
run([
    sys.executable, "-m", "pip", "install", "-q",
    "tiktoken", "pyarrow", "requests", "tokenizers", "huggingface-hub", "safetensors", "matplotlib",
])
run([sys.executable, "-m", "pip", "install", "-q", "rustbpe"], check=False)

# 3. NEVER import jax in the Jupyter kernel so Cloud TPU /tmp/libtpu_lockfile stays free!
#    Run a 4-tier self-healing subprocess probe to unlock Pallas TPU SplashAttention(1024).
PROBE_PY = r"""
import json, os, sys
sys.path.insert(0, "/tmp/autoresearch-xla")
out = {
    "tpu_library_path_env": os.environ.get("TPU_LIBRARY_PATH"),
    "libtpu_pkg_ver": None,
    "libtpu_pkg_path": None,
    "jax_ver": None,
    "num_devices": 0,
    "platform": None,
    "device_kind": None,
    "pallas_ok": False,
    "error": None,
}
try:
    import libtpu  # type: ignore
    out["libtpu_pkg_ver"] = getattr(libtpu, "__version__", None)
    if hasattr(libtpu, "get_library_path"):
        out["libtpu_pkg_path"] = str(libtpu.get_library_path())
except Exception:
    pass
try:
    import jax
    out["jax_ver"] = jax.__version__
    devs = jax.devices()
    out["num_devices"] = len(devs)
    out["platform"] = str(devs[0].platform).lower() if devs else None
    out["device_kind"] = str(getattr(devs[0], "device_kind", "")) if devs else None
    if out["platform"] == "tpu":
        from scale.attention import _probe_pallas_tpu_supported
        out["pallas_ok"] = bool(_probe_pallas_tpu_supported())
except Exception as exc:
    out["error"] = f"{type(exc).__name__}: {exc}"
print("PROBE_JSON=" + json.dumps(out))
"""

def probe_tpu(extra_env=None):
    env = {**os.environ, **(extra_env or {})}
    # Explicitly clear TPU_LIBRARY_PATH if extra_env sets it to None
    if extra_env and "TPU_LIBRARY_PATH" in extra_env and extra_env["TPU_LIBRARY_PATH"] is None:
        env.pop("TPU_LIBRARY_PATH", None)
    r = subprocess.run([sys.executable, "-c", PROBE_PY], cwd=str(REPO_DIR), env=env, capture_output=True, text=True)
    for line in reversed(r.stdout.splitlines()):
        if line.startswith("PROBE_JSON="):
            return json.loads(line[len("PROBE_JSON="):])
    return {"jax_ver": None, "num_devices": 0, "pallas_ok": False, "error": r.stderr[-500:]}

orig_tpu_lib = os.environ.get("TPU_LIBRARY_PATH")
info = probe_tpu()
print(f"[Tier 0] Preinstalled probe: {info}", flush=True)

PALLAS_OK = bool(info.get("pallas_ok")) and info.get("num_devices", 0) >= 1
orig_jax_ver = info.get("jax_ver")
orig_libtpu_ver = info.get("libtpu_pkg_ver")
installed_fresh_libtpu = False

# Tier 1: If site-packages libtpu already exists and differs from TPU_LIBRARY_PATH, point to it!
if not PALLAS_OK and info.get("platform") == "tpu" and info.get("libtpu_pkg_path") and info["libtpu_pkg_path"] != orig_tpu_lib:
    print(f"[Tier 1] Testing installed libtpu package ({info['libtpu_pkg_ver']}) at {info['libtpu_pkg_path']}...", flush=True)
    info1 = probe_tpu({"TPU_LIBRARY_PATH": info["libtpu_pkg_path"]})
    print(f"[Tier 1] Result: {info1}", flush=True)
    if info1.get("pallas_ok") and info1.get("num_devices", 0) == info.get("num_devices", 0):
        os.environ["TPU_LIBRARY_PATH"] = info["libtpu_pkg_path"]
        info = info1
        PALLAS_OK = True

# Tier 2: Upgrade libtpu wheel to match Kaggle's preinstalled JAX (e.g. jax[tpu]==0.11.0 -> libtpu==0.0.44.*)
if not PALLAS_OK and info.get("platform") == "tpu" and orig_jax_ver:
    print(f"[Tier 2] Installing matching libtpu for JAX {orig_jax_ver} via `jax[tpu]=={orig_jax_ver}`...", flush=True)
    r_pip = run([sys.executable, "-m", "pip", "install", "-q", f"jax[tpu]=={orig_jax_ver}"], check=False)
    if r_pip.returncode == 0:
        installed_fresh_libtpu = True
        try:
            pkg_path = subprocess.check_output(
                [sys.executable, "-c", "import libtpu; print(libtpu.get_library_path())"], text=True
            ).strip()
        except Exception as exc:
            print(f"[Tier 2] Could not query libtpu path ({exc}); continuing to Tier 3.", flush=True)
            pkg_path = None
        if pkg_path:
            info2 = probe_tpu({"TPU_LIBRARY_PATH": pkg_path})
            print(f"[Tier 2] Result: {info2}", flush=True)
            if info2.get("pallas_ok") and info2.get("num_devices", 0) >= 1:
                os.environ["TPU_LIBRARY_PATH"] = pkg_path
                info = info2
                PALLAS_OK = True

# Tier 3: Proven fallback — match jax==0.6.2 + jaxlib==0.6.2 to Kaggle's Jun 12 2025 system libtpu.so!
if not PALLAS_OK and info.get("platform") == "tpu":
    print("[Tier 3] Falling back to jax==0.6.2 + jaxlib==0.6.2 matched to Kaggle's Jun-2025 libtpu...", flush=True)
    if orig_tpu_lib:
        os.environ["TPU_LIBRARY_PATH"] = orig_tpu_lib
    else:
        os.environ.pop("TPU_LIBRARY_PATH", None)
    if installed_fresh_libtpu and not orig_libtpu_ver:
        run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "libtpu"], check=False)
    run([sys.executable, "-m", "pip", "install", "-q", "jax==0.6.2", "jaxlib==0.6.2"], check=False)
    info3 = probe_tpu({"TPU_LIBRARY_PATH": orig_tpu_lib})
    print(f"[Tier 3] Result: {info3}", flush=True)
    if info3.get("pallas_ok") and info3.get("num_devices", 0) >= 1:
        info = info3
        PALLAS_OK = True

# 4. Probe all 10 MaxText FSDP LIBTPU_INIT_ARGS flags (auto-filtering any unsupported flags)
_tree = ast.parse((REPO_DIR / "scale" / "pretrain.py").read_text(encoding="utf-8"))
_DEFAULT_TPU_XLA_FLAGS = ""
for _node in _tree.body:
    if isinstance(_node, ast.Assign):
        for _t in _node.targets:
            if isinstance(_t, ast.Name) and _t.id == "_DEFAULT_TPU_XLA_FLAGS":
                _DEFAULT_TPU_XLA_FLAGS = ast.literal_eval(_node.value)
assert _DEFAULT_TPU_XLA_FLAGS, "Failed to parse _DEFAULT_TPU_XLA_FLAGS"

if info.get("platform") == "tpu":
    chk = subprocess.run(
        [sys.executable, "-c", "import jax; print(len(jax.devices()))"],
        cwd=str(REPO_DIR),
        env={**os.environ, "LIBTPU_INIT_ARGS": _DEFAULT_TPU_XLA_FLAGS},
        capture_output=True,
        text=True,
    )
    if chk.returncode == 0:
        os.environ["LIBTPU_INIT_ARGS"] = _DEFAULT_TPU_XLA_FLAGS
        print(f"[XLA] All MaxText FSDP LIBTPU_INIT_ARGS enabled ({len(_DEFAULT_TPU_XLA_FLAGS.split())} flags).", flush=True)
    else:
        accepted = []
        for fl in _DEFAULT_TPU_XLA_FLAGS.split():
            cand = " ".join(accepted + [fl])
            r_fl = subprocess.run(
                [sys.executable, "-c", "import jax; print(len(jax.devices()))"],
                cwd=str(REPO_DIR),
                env={**os.environ, "LIBTPU_INIT_ARGS": cand},
                capture_output=True,
                text=True,
            )
            if r_fl.returncode == 0:
                accepted.append(fl)
            else:
                print(f"  Skipping unsupported libtpu flag: {fl}", flush=True)
        os.environ["LIBTPU_INIT_ARGS"] = " ".join(accepted)
        print(f"[XLA] Enabled {len(accepted)} compatible LIBTPU_INIT_ARGS flags.", flush=True)

print(
    f"\n=== ENV READY ===\n"
    f"  JAX={info.get('jax_ver')} | {info.get('device_kind')} x{info.get('num_devices')} ({info.get('platform')})\n"
    f"  PALLAS_OK={PALLAS_OK} | TPU_LIBRARY_PATH={os.environ.get('TPU_LIBRARY_PATH', '(default)')}",
    flush=True,
)
run(["df", "-h", "/kaggle/working", "/tmp"], check=False)


In [ ]:
import json
import os
from pathlib import Path

# ---- Training & Checkpoint Configuration (Round-5 Champion 1b-ve, 1B-Safe Stable LRs) ----
HF_REPO_NAME = "autoresearch-1b-ve"   # Private repo: <your-hf-username>/autoresearch-1b-ve
PRESET = "1b-ve"                      # 1.456B total / 1.120B active, Split-Muon ON, MTP OFF
DATASET = "mix-1b"                    # 65% ClimbMix + 15% CodeParrot + 10% FineMath-4plus + 10% Cosmopedia-v2
TOTAL_TOKENS = 20_000_000_000         # 20B tokens (Chinchilla-optimal)
TOTAL_BATCH_SIZE = 524_288            # 512K tokens/step
PHASE = "stable"                      # Keep "stable" across 8.2h sessions; switch to "decay" for final ~2B cooldown!
WARMUP_STEPS = 1000                   # Smooth 524M-token warmup (prevents post-warmup gnorm spike)
DECAY_STEPS = 4000
MATRIX_LR = 0.015                     # 1B-safe Split-Muon LR (Session 1 0.04 was 2.7x too hot at C=2048!)
EMBEDDING_LR = 0.25                   # Width-scaled by (2048/768)^-0.5 -> 0.153 eff LR
UNEMBEDDING_LR = 0.003                # Gentle lm_head LR to keep logit z-loss calm
SCALAR_LR = 0.1                       # Gentle LR for x0_lambdas, qk_gain, canon_a/c
WEIGHT_DECAY = 0.1                    # Standard 1B cautious Muon WD (was 0.03 in Session 1)
FORCE_FRESH_START = False             # Set True to force step-0 restart regardless of HF checkpoint health
MAX_HOURS = 8.2                       # Safe 48-min buffer before Kaggle 9.0h TPU timeout!
LOG_EVERY = 10
EVAL_EVERY = 250                      # Pinned ClimbMix shard_06542 val_bpb
SAMPLE_EVERY = 500                    # Live sample right AFTER every checkpoint save
SAVE_EVERY = 500                      # Atomic HF Hub upload every ~15-20 min (262M tokens)
KEEP_LAST_CHECKPOINTS = 1             # MANDATORY on Kaggle 19.5 GB disk (6.77 GB steady / 13.54 GB atomic peak)
KEEP_SHARDS = 2
CHECKPOINT_DIR = "/kaggle/working/ckpt_1b" if Path("/kaggle/working").exists() else "./ckpt_1b"
SHARD_CACHE_DIR = "/tmp/shards_mix1b"
PROFILE_DIR = "/kaggle/working/xprof_session2" if Path("/kaggle/working").exists() else "./xprof_session2"
PROFILE_START_STEP = 3
PROFILE_STEPS = 2
# ------------------------------------------------------------------------------------------

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient  # type: ignore
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as exc:
        print(f"[HF] Kaggle Secret HF_TOKEN not attached ({exc})", flush=True)

SYNC_BACKEND = "none"
REMOTE_TARGET = None
USE_NO_RESUME = bool(FORCE_FRESH_START)

if hf_token:
    hf_token = str(hf_token).strip()
    os.environ["HF_TOKEN"] = hf_token
    try:
        from huggingface_hub import HfApi, hf_hub_download
        api = HfApi(token=hf_token)
        hf_user = api.whoami()["name"]
        REMOTE_TARGET = f"{hf_user}/{HF_REPO_NAME}"
        api.create_repo(repo_id=REMOTE_TARGET, repo_type="model", private=True, exist_ok=True)
        SYNC_BACKEND = "hf"
        files = set(api.list_repo_files(repo_id=REMOTE_TARGET, repo_type="model"))
        if "latest/trainer_state.json" in files:
            ts_path = hf_hub_download(repo_id=REMOTE_TARGET, filename="latest/trainer_state.json", token=hf_token)
            ts = json.loads(Path(ts_path).read_text(encoding="utf-8"))
            saved_step = ts.get("step", 0)
            saved_best_bpb = ts.get("best_val_bpb", None)
            print(f"[HF] Found existing checkpoint on hf://{REMOTE_TARGET} at step={saved_step} (best_val_bpb={saved_best_bpb})", flush=True)

            # Inspect metrics.jsonl (if uploaded) to verify checkpoint was healthy BEFORE resuming!
            healthy = True
            if "metrics.jsonl" in files or "latest/metrics.jsonl" in files:
                m_file = "latest/metrics.jsonl" if "latest/metrics.jsonl" in files else "metrics.jsonl"
                mp = hf_hub_download(repo_id=REMOTE_TARGET, filename=m_file, token=hf_token)
                rows = [json.loads(ln) for ln in Path(mp).read_text(encoding="utf-8").splitlines() if ln.strip()]
                evals = [r for r in rows if "val_bpb" in r and r.get("step", 0) <= saved_step]
                trains = [r for r in rows if ("grad_norm" in r or "gnorm" in r) and r.get("step", 0) <= saved_step]
                print("[HF] Saved checkpoint eval history:", [(r["step"], r["val_bpb"]) for r in evals[-6:]], flush=True)
                if evals:
                    best_bpb = min(r["val_bpb"] for r in evals)
                    last_bpb = evals[-1]["val_bpb"]
                    if last_bpb > best_bpb + 0.015:
                        print(f"[HF] WARNING: Saved step {saved_step} val_bpb={last_bpb:.4f} already regressed vs best={best_bpb:.4f}!", flush=True)
                        healthy = False
                if trains:
                    recent_gn = [r.get("grad_norm", r.get("gnorm", 0.0)) for r in trains[-5:]]
                    print(f"[HF] Recent grad_norms before step {saved_step}: {recent_gn}", flush=True)
                    if sum(recent_gn) / len(recent_gn) > 0.8:
                        print(f"[HF] WARNING: Saved step {saved_step} avg grad_norm > 0.8 — unsafe to resume!", flush=True)
                        healthy = False

            if FORCE_FRESH_START:
                print("[HF] FORCE_FRESH_START=True -> starting clean from step 0 (--no-resume).", flush=True)
                USE_NO_RESUME = True
            elif healthy:
                print(f"[HF] Checkpoint step {saved_step} is HEALTHY -> WILL AUTO-RESUME with 1B-safe LRs (matrix_lr={MATRIX_LR}, wd={WEIGHT_DECAY})!", flush=True)
                USE_NO_RESUME = False
            else:
                print("[HF] Degraded Session-1 checkpoint detected -> auto-switching to CLEAN step-0 start (--no-resume)!", flush=True)
                USE_NO_RESUME = True
        else:
            print(f"[HF] Authenticated as '{hf_user}' -> PRIVATE repo hf://{REMOTE_TARGET} (empty — starting fresh from step 0!)", flush=True)
    except Exception as exc:
        print(f"[HF] WARNING: HF Hub auth/repo preflight failed ({exc}). Falling back to local {CHECKPOINT_DIR}.", flush=True)
        SYNC_BACKEND = "none"
        REMOTE_TARGET = None
else:
    print("=" * 78, flush=True)
    print("WARNING: HF_TOKEN secret not found in Kaggle Secrets!", flush=True)
    print(f"Proceeding with LOCAL backup ONLY ({CHECKPOINT_DIR}).", flush=True)
    print("=" * 78, flush=True)

print(
    f"Config: backend={SYNC_BACKEND} | remote={REMOTE_TARGET} | no_resume={USE_NO_RESUME} | "
    f"matrix_lr={MATRIX_LR} | emb_lr={EMBEDDING_LR} | unemb_lr={UNEMBEDDING_LR} | scalar_lr={SCALAR_LR} | wd={WEIGHT_DECAY}",
    flush=True,
)


In [ ]:
import json
import math
import os
import re
import subprocess
import sys
import time
from pathlib import Path

REPO_DIR = Path("/tmp/autoresearch-xla")
OUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

RESULT_RE = re.compile(
    r"\[Bench RESULT\] compile=([\d.]+)s \| step_dt=([\d.]+)ms \| "
    r"tok/s=([\d,]+) \| MFU=([\d.]+)% \| peak_mem=([\d.]+)MB \| loss=([^\s|]+)"
)
ATTN_RE = re.compile(r"attn=([a-z]+)")

SESSION1_BASELINE_TOK_S = 87_631  # Live Kaggle v5e-8 main (4342b94): 5,983 ms/step, 38.6% MFU

# Ordered fastest -> safest! Default exp/xla-vendetta uses real-silicon-verified pure-XLA + SplashAttention
# + Layer-Sharded Split-Muon + Contracting-Axis FSDP + Replicated Small Keys + Hidden-Sharded Embeds.
if PALLAS_OK:
    CANDIDATES = [
        ("vendetta_b4_sac_splash", 4, "save_all_compute", "splash", []),
        ("vendetta_b4_save_attn_splash", 4, "save_attn", "splash", []),
    ]
    MIN_TOK_PER_SEC = 60_000
    FAST_ACCEPT_TOK_S = 85_000
else:
    CANDIDATES = [
        ("vendetta_b4_xla", 4, "save_attn", "xla", []),
        ("vendetta_b2_xla", 2, "save_attn", "xla", []),
    ]
    MIN_TOK_PER_SEC = 25_000
    FAST_ACCEPT_TOK_S = 25_000

COMMON_ARCH_ARGS = [
    "--preset", PRESET,
    "--total-batch-size", str(TOTAL_BATCH_SIZE),
    "--splash-block-size", "1024",
    "--loss-chunk-size", "4096",
    "--param-dtype", "bfloat16",
    "--opt-dtype", "bfloat16",
    "--matmul-quant", "int8-all",
    "--muon-vmap",
    "--matrix-lr", str(MATRIX_LR),
    "--embedding-lr", str(EMBEDDING_LR),
    "--unembedding-lr", str(UNEMBEDDING_LR),
    "--scalar-lr", str(SCALAR_LR),
    "--weight-decay", str(WEIGHT_DECAY),
]

def run_preflight_candidate(name, per_dev_b, remat, attn, extra_args):
    print(f"\n================ [STAGE 1 PREFLIGHT: {name}] ================", flush=True)
    cmd = [
        sys.executable, "-m", "scale.pretrain",
        *COMMON_ARCH_ARGS,
        *extra_args,
        "--per-device-batch", str(per_dev_b),
        "--remat-policy", remat,
        "--attn-impl", attn,
        "--benchmark-only",
        "--bench-steps", "6",
    ]
    t0 = time.time()
    proc = subprocess.run(
        cmd,
        cwd=str(REPO_DIR),
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
        capture_output=True,
        text=True,
    )
    wall = round(time.time() - t0, 1)
    print(proc.stdout, flush=True)
    entry = {
        "name": name,
        "per_device_batch": per_dev_b,
        "remat_policy": remat,
        "attn_impl": attn,
        "extra_args": list(extra_args),
        "wall_s": wall,
        "passed": False,
        "reason": None,
    }
    if proc.returncode != 0:
        tail = "\n".join(proc.stderr.splitlines()[-15:])
        print(f"!!! [{name}] FAILED (rc={proc.returncode}, {wall}s):\n{tail}", flush=True)
        entry["reason"] = f"rc={proc.returncode}: {tail[-300:]}"
        return entry

    m = RESULT_RE.search(proc.stdout)
    m_attn = ATTN_RE.search(proc.stdout)
    if not m:
        entry["reason"] = "Could not parse [Bench RESULT]"
        return entry

    compile_s = float(m.group(1))
    step_ms = float(m.group(2))
    tok_s = int(m.group(3).replace(",", ""))
    mfu = float(m.group(4))
    peak_mem = float(m.group(5))
    try:
        loss = float(m.group(6))
    except ValueError:
        loss = float("nan")
    attn_used = m_attn.group(1) if m_attn else attn

    entry.update({
        "compile_s": compile_s,
        "step_ms": step_ms,
        "tok_per_sec": tok_s,
        "mfu_percent": mfu,
        "peak_mem_mb": peak_mem,
        "loss": loss if math.isfinite(loss) else None,
        "attn_used": attn_used,
        "speedup_vs_s1": round(tok_s / SESSION1_BASELINE_TOK_S, 3),
    })

    if not math.isfinite(loss) or loss <= 0.0 or loss >= 15.0:
        entry["reason"] = f"Bad loss={loss}"
    elif PALLAS_OK and attn_used != "splash":
        entry["reason"] = f"Expected attn=splash, got attn={attn_used}"
    elif tok_s < MIN_TOK_PER_SEC:
        entry["reason"] = f"Throughput {tok_s:,} tok/s below gate {MIN_TOK_PER_SEC:,}"
    elif peak_mem > 15_200.0:
        entry["reason"] = f"Peak HBM {peak_mem:.0f} MB exceeds 15,200 MB safety ceiling"
    else:
        entry["passed"] = True
        entry["reason"] = "OK"

    gate_str = "PASS" if entry["passed"] else "FAIL"
    print(f"[{name}] gate={gate_str} ({entry['reason']}) | vs S1: {entry.get('speedup_vs_s1', 0):.3f}x", flush=True)
    return entry

preflight_results = []
passing = []
for idx, c in enumerate(CANDIDATES):
    r = run_preflight_candidate(*c)
    preflight_results.append(r)
    if r["passed"]:
        passing.append(r)
        if idx == 0 and r["tok_per_sec"] >= FAST_ACCEPT_TOK_S:
            print(f"[Preflight] Primary passed ({r['tok_per_sec']:,} tok/s, {r['speedup_vs_s1']:.3f}x vs S1) — promoting immediately!", flush=True)
            break
        if idx >= 1:
            break

(OUT_ROOT / "preflight_summary.json").write_text(json.dumps(preflight_results, indent=2) + "\n")

if not passing:
    raise RuntimeError(
        "STAGE 1 PREFLIGHT FAILED on all candidates — aborting cleanly BEFORE touching HF Hub checkpoints! "
        f"Details: {json.dumps(preflight_results, indent=2)}"
    )

WINNER = max(passing, key=lambda r: r["tok_per_sec"])
est_tok_session_b = WINNER["tok_per_sec"] * (MAX_HOURS * 3600.0) * 0.92 / 1e9
est_steps_session = int((est_tok_session_b * 1e9) // TOTAL_BATCH_SIZE)

print("\n" + "=" * 78, flush=True)
print("STAGE 1 PREFLIGHT PASSED — AUTO-PROMOTING TO 8.2h STABLE 1b-ve TRAINING!", flush=True)
print(
    f"  Winner:      {WINNER['name']} (b={WINNER['per_device_batch']}, remat={WINNER['remat_policy']}, attn={WINNER['attn_used']})\n"
    f"  Speed:       {WINNER['tok_per_sec']:,} tok/s ({WINNER['speedup_vs_s1']:.3f}x vs S1!) | MFU={WINNER['mfu_percent']:.2f}% | step={WINNER['step_ms']:.1f}ms\n"
    f"  Sanity:      loss={WINNER['loss']:.4f} | compile={WINNER['compile_s']:.1f}s | peak_mem={WINNER['peak_mem_mb']:.0f}MB\n"
    f"  LRs:         matrix={MATRIX_LR} emb={EMBEDDING_LR} unemb={UNEMBEDDING_LR} scalar={SCALAR_LR} wd={WEIGHT_DECAY} warmup={WARMUP_STEPS}\n"
    f"  Est/Session: ~{est_tok_session_b:.2f}B tokens (~{est_steps_session:,} steps) in {MAX_HOURS:.1f}h",
    flush=True,
)
print("=" * 78, flush=True)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/tmp/autoresearch-xla")

cmd = [
    sys.executable, "-m", "scale.pretrain",
    "--preset", PRESET,
    "--dataset", DATASET,
    "--total-tokens", str(TOTAL_TOKENS),
    "--total-batch-size", str(TOTAL_BATCH_SIZE),
    "--per-device-batch", str(WINNER["per_device_batch"]),
    "--remat-policy", str(WINNER["remat_policy"]),
    "--attn-impl", str(WINNER["attn_used"]),
    "--splash-block-size", "1024",
    "--loss-chunk-size", "4096",
    "--param-dtype", "bfloat16",
    "--opt-dtype", "bfloat16",
    "--matmul-quant", "int8-all",
    "--muon-vmap",
    "--matrix-lr", str(MATRIX_LR),
    "--embedding-lr", str(EMBEDDING_LR),
    "--unembedding-lr", str(UNEMBEDDING_LR),
    "--scalar-lr", str(SCALAR_LR),
    "--weight-decay", str(WEIGHT_DECAY),
    "--phase", PHASE,
    "--warmup-steps", str(WARMUP_STEPS),
    "--decay-steps", str(DECAY_STEPS),
    "--log-every", str(LOG_EVERY),
    "--eval-every", str(EVAL_EVERY),
    "--sample-every", str(SAMPLE_EVERY),
    "--save-every", str(SAVE_EVERY),
    "--keep-last-checkpoints", str(KEEP_LAST_CHECKPOINTS),
    "--keep-shards", str(KEEP_SHARDS),
    "--checkpoint-dir", CHECKPOINT_DIR,
    "--shard-cache-dir", SHARD_CACHE_DIR,
    "--sync-backend", SYNC_BACKEND,
    "--max-hours", str(MAX_HOURS),
    "--profile-dir", PROFILE_DIR,
    "--profile-start-step", str(PROFILE_START_STEP),
    "--profile-steps", str(PROFILE_STEPS),
    *WINNER.get("extra_args", []),
]
if REMOTE_TARGET:
    cmd += ["--remote-target", REMOTE_TARGET]
if USE_NO_RESUME:
    cmd += ["--no-resume"]

print("Launching Stage 2 Training:\n  " + " ".join(cmd) + "\n", flush=True)
proc = subprocess.Popen(
    cmd,
    cwd=str(REPO_DIR),
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"Stage 2 Pretraining exited with code {rc}")


In [ ]:
import json
import subprocess
from pathlib import Path
import matplotlib.pyplot as plt

OUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
ckpt_root = Path(CHECKPOINT_DIR)
metrics_file = ckpt_root / "metrics.jsonl"

summary = {
    "preset": PRESET,
    "dataset": DATASET,
    "phase": PHASE,
    "winner_profile": WINNER,
    "sync_backend": SYNC_BACKEND,
    "remote_target": REMOTE_TARGET,
    "hf_url": f"https://huggingface.co/{REMOTE_TARGET}" if (SYNC_BACKEND == "hf" and REMOTE_TARGET) else None,
}

if metrics_file.exists():
    rows = [json.loads(line) for line in metrics_file.read_text().splitlines() if line.strip()]
    train_rows = [r for r in rows if "loss" in r]
    eval_rows = [r for r in rows if "val_bpb" in r]
    steady_rows = [r for r in train_rows if r.get("step", 0) >= 10]

    if train_rows:
        summary["final_step"] = train_rows[-1]["step"]
        summary["tokens_seen_B"] = round(train_rows[-1]["tokens_seen"] / 1e9, 4)
        summary["final_train_loss"] = train_rows[-1]["loss"]
    if steady_rows:
        summary["avg_tok_per_sec"] = round(sum(r["tok_per_sec"] for r in steady_rows) / len(steady_rows), 1)
        summary["avg_mfu_percent"] = round(sum(r["mfu_percent"] for r in steady_rows) / len(steady_rows), 2)
    if eval_rows:
        summary["best_val_bpb"] = min(r["val_bpb"] for r in eval_rows)
        summary["final_val_bpb"] = eval_rows[-1]["val_bpb"]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), dpi=130)
    if train_rows:
        axes[0].plot([r["tokens_seen"] / 1e9 for r in train_rows], [r["loss"] for r in train_rows], color="#38bdf8")
        axes[0].set_title("1b-ve Training Loss vs Tokens (B)")
        axes[0].set_xlabel("Tokens Seen (Billions)")
        axes[0].set_ylabel("Cross-Entropy Loss (nats)")
        axes[0].grid(alpha=0.3)
    if eval_rows:
        axes[1].plot([r["tokens_seen"] / 1e9 for r in eval_rows], [r["val_bpb"] for r in eval_rows], marker="o", color="#22c55e")
        axes[1].set_title("1b-ve Validation val_bpb vs Tokens (B)")
        axes[1].set_xlabel("Tokens Seen (Billions)")
        axes[1].set_ylabel("val_bpb (lower is better)")
        axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_ROOT / "training_curve.png", dpi=130)
    plt.show()
else:
    print("No metrics.jsonl found.", flush=True)

(OUT_ROOT / "session_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print("\n=== SESSION SUMMARY ===", flush=True)
print(json.dumps(summary, indent=2), flush=True)
subprocess.run(["du", "-sh", str(ckpt_root)], check=False)
subprocess.run(["df", "-h", "/kaggle/working"], check=False)
